In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from datetime import datetime
import subprocess
import time
import re
import glob
import json

import psutil
import os

from google.colab import auth

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

from huggingface_hub import login

import random
import numpy as np
import pickle
from dataclasses import dataclass
from datasets import load_dataset, DatasetDict, Dataset


import requests
import asyncio
import aiohttp
import itertools
import nest_asyncio

from tqdm.notebook import tqdm
from typing import List, Dict
from math import comb
import re


In [ ]:
# This downalods the Yang model, Dataset, ...
auth.authenticate_user()

drive = build("drive", "v3")


def download_from_drive(file_id, out_name):
  request = drive.files().get_media(fileId=file_id)
  fh = io.FileIO(out_name, "wb")
  downloader = MediaIoBaseDownload(fh, request)
  done = False
  while not done:
      status, done = downloader.next_chunk()

  print("Downloaded", out_name)
  print("First line:", open(out_name, "r", encoding="utf-8").readline()[:300])
  print("-" * 100)

downloads = [ {"id": "17c2o91SQslzeWiAvO-RRo9FmNBq6ZW3v", "out": "config.json" },
             {"id": "1eDA2jnIfHfwYM_fJ_UeVKG5HqyKiBG8r", "out": "example_case.txt"},
              {"id": "1hSyVH_WiXZXCYeTNd9BW_If_RLsGi1gx", "out": "network_config.yang"},
              {"id": "158Ape9lDL6BM9JL73BQYZs_pogqcHISZ", "out": "FINAL_p4_ds.jsonl"}]


for download in downloads:
  download_from_drive(download["id"], download["out"])

Downloaded config.json
First line: {

----------------------------------------------------------------------------------------------------
Downloaded example_case.txt
First line: === INTENT ===

----------------------------------------------------------------------------------------------------
Downloaded network_config.yang
First line: module network-config {

----------------------------------------------------------------------------------------------------
Downloaded FINAL_p4_ds.jsonl
First line: {"repo_name":"abhikjain360\/sdn-project","desc":null,"repo_path":"raw-repositories\/abhikjain360__sdn-project","p4_version":"p4_16","p4_file_path":"raw-repositories\/abhikjain360__sdn-project\/swtichtree.p4","readme_path":"raw-repositories\/abhikjain360__sdn-project\/README.md","license":"mit","infe
----------------------------------------------------------------------------------------------------


In [ ]:
def load_ds(path: str = "FINAL_p4_ds.jsonl"):
  dataset = load_dataset("json", data_files=path)
  return dataset

In [ ]:
def train_val_test_split(dataset: Dataset, train_size: float = 0.85, val_size: float = 0.05, test_size: float = 0.10):
  assert train_size + val_size + test_size == 1.0

  X = dataset.train_test_split(train_size=train_size)

  X2 = X["test"].train_test_split(train_size = (val_size / (1 - train_size)) )

  return DatasetDict(
    {
      "train": X["train"],
      "validation": X2["train"],
      "test": X2["test"]
    }
  )

In [ ]:


@dataclass
class FewShotExample:
  annotation: str
  code: str



# if tokenizer is None, then it's assumed that chat template and stuff will be applied on the API (I.E. WE'RE USING THIS FOR API)
class PromptBuilder:
  def __init__(self, tokenizer, few_shot_examples: list[FewShotExample] | None = None):
    self.tokenizer = tokenizer
    self.few_shot_examples = few_shot_examples

  # this function needs refinement in case each intent has different yang model/data
  def get_yang_model(self, default_yang_model_path="network_config.yang"):
    # A FAIRE: handle file not exist
    with open(default_yang_model_path, "r") as f:
      yang_content = f.read()

    return yang_content


  # this function needs refinement in case each intent has different yang model/data
  def get_yang_data(self, default_yang_data_path="config.json"):
    # A FAIRE: handle file not exist
    with open(default_yang_data_path, "r") as f:
      yang_data = f.read()

    return yang_data


  def create_detailed_prompt(self, user_intent):
    yang_model: str = self.get_yang_model()
    yang_data: str = self.get_yang_data()
    few_shot_examples: list[FewShotExample] = self.few_shot_examples

    """ [TODO 2]
    Make sure that whatever is in the example_prompt_format string matches what you did to test the previous LLM. That is the
    exact verbatim text. For the few-show examples, also use the exact ones you used from before, you will be passing the in the very last cell

    You can look inspect example_prompt_format variable which will show you what the LLM gets as it's final prompt
    """

    example_prompt_format = f"""
You are generating one compilable P4_16 program for BMv2 v1model.

TASK:
Implement the network intent: {user_intent}

YANG MODEL SCHEMA:
The following YANG model defines the data structure and constraints:
```yang
{yang_model}
```

CURRENT NETWORK CONFIGURATION:
```json
{yang_data}
```

{"EXAMPLES:" if few_shot_examples else ""}
{"Here are examples of similar network intents and their P4 implementations:" if few_shot_examples else ""}

{chr(10).join([
    f"Example {i+1}:{chr(10)}Intent: {example.annotation}{chr(10)}Implementation:{chr(10)}```p4{chr(10)}{example.code}{chr(10)}```{chr(10)}"
    for i, example in enumerate(few_shot_examples)
]) if few_shot_examples else ""}

REQUIREMENTS:
- You must target BMv2 with v1model architecture.
- You must include all required components: headers, parser, ingress/egress controls, deparser, and V1Switch main block.
- You must ensure compatibility with the p4c compiler.
- You must use the YANG model schema to understand data structures.
- You must consider the current network configuration when implementing logic.
- You must follow the patterns shown in the examples above.
- You must output exactly one complete, compilable program.
- You must not include prose, comments, or markdown in the output.
- You must start your reply with `<p4>` on the first line and end with `</p4>` on the last line.

Generate the P4 program now:
"""

    return example_prompt_format

  def build_prompts(self, user_intents: list[str]):
    message_groups = [
        [
            # !!! Do not change the system prompt! !!!
            {"role": "system", "content": """You are a P4 code generator. Respond with one compilable P4_16 program wrapped in <p4>...</p4> tags. Do not include any explanation or comments. Always include headers, parser, ingress/egress controls, deparser, and main block (e.g., V1Switch(...)). Target BMv2 with v1model and ensure the output works with p4c."""},
            {"role": "user", "content": self.create_detailed_prompt(user_intent)}
        ]
        for user_intent in user_intents
    ]

    if self.tokenizer is not None:
      formatted_prompts = [ self.tokenizer.apply_chat_template(message_group, tokenize=False, add_generation_prompt=True) for message_group in message_groups ]

      if "<p4>" in self.tokenizer.additional_special_tokens:
        formatted_prompts = [formatted_prompt + "<p4>"  for formatted_prompt in formatted_prompts ]

      return formatted_prompts
    else:
      return message_groups


  def __call__(self, user_intents: list[str]):
    return self.build_prompts(user_intents)


In [ ]:

# this thingy is added because colab itself is running in async container, so gotta do this thing
nest_asyncio.apply()


@dataclass
class EvaluationTask:
    prompt: str
    code: str


class EndPoint:
    def __init__(self, ip: str, port: int, url: str):
        self.ip = ip
        self.port = port
        self.url = url

# This class handles talking to the server, i.e validing P4 code, ...
class Evaluator:
    def __init__(self, max_workers=os.cpu_count()):
        self.max_workers = max_workers

        self.compiler_end_point = EndPoint("34.132.100.178", 8000, "/validate")
        self.p4_test_gen_end_point = EndPoint("34.132.100.178", 8000, "/validate")


    @staticmethod
    def _sanitize_p4_outputs(example: str):
        start_pattern = "#include"
        end_pattern = "main;"

        start_idx = example.find(start_pattern)
        if start_idx != -1:
            end_idx = example.find(end_pattern, start_idx)
            if end_idx != -1:
                end_idx += len(end_pattern)
                return example[start_idx:end_idx].strip()

        match = re.search(r"<p4>(.*?)</p4>", example, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
        return ""


    async def _evaluate_single_p4_code(self, task: EvaluationTask, session, pbar, retries=20):
        endpoint = self.p4_test_gen_end_point
        req_url = f"http://{endpoint.ip}:{endpoint.port}{endpoint.url}"

        for i in range(retries):
          try:
            async with session.post(url=req_url, json={"code": self._sanitize_p4_outputs(task.code)}) as response:
                result = await response.json()
                pbar.update(1)
                return {task.prompt: result}
          except Exception as e:
            print('expection occured, retrying to validate!')

    async def _evaluate_batch(self, batch: List[EvaluationTask], pbar, session):
        tasks = [self._evaluate_single_p4_code(task, session, pbar) for task in batch]
        return await asyncio.gather(*tasks)

    async def _evaluate_all_batches(self, all_tasks: List[EvaluationTask], pbar, batch_size):
        results = []
        async with aiohttp.ClientSession() as session:
            for i in range(0, len(all_tasks), batch_size):
                batch = all_tasks[i:i + batch_size]
                batch_results = await self._evaluate_batch(batch, pbar, session)
                results.extend(batch_results)
        return results


    async def _evaluate_from_generator(self, generator, batch_size=24):
      result = []
      async with aiohttp.ClientSession() as session:
        pbar = tqdm(desc="Validating", unit="gen", position=1)

        async for batch in generator:
          eval_tasks = []
          for result_dict in batch:
            for prompt, code in result_dict.items():
              eval_tasks.append(EvaluationTask(prompt, code))

          batch_results = await self._evaluate_batch(eval_tasks, pbar, session)
          result += batch_results

      return result

    def _has_passed_test_cases(self, result):
      key = list(result.keys())[0]
      value = result[key]
      return value["test_cases"]


    def evaluate_generations(self, results, prompts, batch_size=24):
        evaluation_tasks = []
        for prompt, result_group in zip(prompts, results):
            for result in result_group.outputs:
                evaluation_tasks.append(EvaluationTask(prompt, result.cleaned_text))

        print("Constructed evaluation tasks array, starting eval!")
        pbar = tqdm(total=len(evaluation_tasks), desc="Validating")
        loop = asyncio.get_event_loop()
        all_results = loop.run_until_complete(self._evaluate_all_batches(evaluation_tasks, pbar, batch_size))
        pbar.close()

        pass_rate_dict = {}

        for result in all_results:
          key = list(result.keys())[0]
          pass_rate_dict[key] = pass_rate_dict.get(key, 0) + int(self._has_passed_test_cases(result))

        return pass_rate_dict, all_results




    def evaluate_generations_generator(self, generator, prompts, batch_size=24):
        async def runner():
            all_results = await self._evaluate_from_generator(generator, batch_size)

            pass_rate_dict = {}
            for result in all_results:
                key = list(result.keys())[0]
                passed = int(self._has_passed_test_cases(result))
                pass_rate_dict[key] = pass_rate_dict.get(key, 0) + passed

            return pass_rate_dict, all_results

        loop = asyncio.get_event_loop()
        return loop.run_until_complete(runner())




    def pass_at_k(self, prompt_counts: List[Dict[str, int]], ks=[1, 10, 100], n=100):
        ret = {}
        for k in ks:
            sum_val = 0
            total = 0
            for prompt, c in prompt_counts.items():
                sum_val += 1 - (comb(n - c, k) / comb(n, k))
                total += 1
            ret[k] = sum_val / total
        return ret

    # all_results is a list of form [ "prompt_generation_1": { compiled: True, "passed": False, "stderr": "...", "stdout": ... } ]
    def compile_rate_at_1(self, all_results):
      compiled = 0
      total = len(all_results)

      for result in all_results:
        result_key = list(result.keys())[0]
        if result[result_key]["compiled"]:
          compiled += 1

      return compiled / total


In [ ]:

validation_ds = train_val_test_split(load_ds()["train"])["test"]
test_intents = list(validation_ds["annotation"])

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
class OpenRouterGenerator:

  def __init__(self, prompt_builder, model_name="deepseek/deepseek-r1:free", token="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"):
    self.model_name = model_name
    self.prompt_builder = prompt_builder

    self.url = "https://openrouter.ai/api/v1/chat/completions"
    self.headers = {
        "Authorization": f"Bearer {token}",
        "HTTP-Referer": os.environ.get("APP_REFERER", "http://localhost"),
        "X-Title": "IBN",
    }


  async def _generate_single_response(self, prompt, session, pbar, retries=20):

    """ [TODO 3]
    make sure the parameters match!!!! Thanks to Gwen, I totally forgot about this, LMAO :-)
    """
    payload = {
      "model": self.model_name,
      "messages": self.prompt_builder([prompt])[0],
      "max_output_tokens": 4096,
      "temperature": 0.9,
      "top_p": 0.95,
      "reasoning": {
        "effort": "medium"
      }
    }

    for i in range(retries):
      try:
        async with session.post(self.url, json=payload, headers=self.headers) as resp:
          if resp.status == 200:
            data = await resp.json()
            if data == None:
              print(f"Generation failed, but status == 200 with {resp.status}. Retrying {i+1}/{retries}")
              continue

            pbar.update()

            return {prompt: data["choices"][0]["message"]["content"]}
          else:
            print(f"Generation failed with {resp.status}. Retrying {i+1}/{retries}")
      except Exception as e:
        print(f"Generation failed due to connection error {e}. Retrying {i+1}/{retries}")

    return {prompt: f"<FAIL> GENERATION FAILED AFTER {retries + 1} RETRIES. </FAIL>"}


  async def _async_generate(self, prompts, n, batch_size=2):
    async with aiohttp.ClientSession() as session:
      prompt_pool = list(itertools.chain.from_iterable([([prompt] * n) for prompt in prompts ]))

      with tqdm(total=len(prompt_pool), desc="generating...", position=1) as pbar:
        results = []
        for i in range(0, len(prompt_pool), batch_size):
          batch_results = await asyncio.gather(*[self._generate_single_response(prompt, session, pbar) for prompt in prompt_pool[i:i+batch_size] ])
          results += batch_results

          with open("generations.jsonl", "a", encoding="utf-8") as f:
            for item in batch_results:
              f.write(json.dumps(item, ensure_ascii=False) + "\n")
          yield batch_results




In [ ]:

# """ [TODO 1]
# So the few_shot_examples array should be populated with FewShotExample objects that are of (intent, corresponding code) format.
# you have calls to this class in other notebooks for pass@k, which are the exact examples you wanna use.

# """
# prompt_builder = PromptBuilder(tokenizer=None, few_shot_examples=[FewShotExample("basic ip4 forwarding", code)])


# """ [TODO 3]
# Choose a model!!!
# One of:

# and replace the variable model
# """

# available_models = [" PLACEHOLDER, CHOOSE AN ACTUAL ENTRY ", "openai/gpt-5", "meta-llama/llama-4-maverick", "qwen/qwen3-max", "google/gemini-2.5-flash"]

# MODEL_NAME = available_models[4]

# gen = OpenRouterGenerator(prompt_builder, model_name=MODEL_NAME, token="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")
# evaluator = Evaluator()

# prompts = list(validation_ds["annotation"])

# NUM_GENERATIONS = 10 # how many generations per prompt

# generator = gen._async_generate(prompts , n=NUM_GENERATIONS, batch_size=64)

# """ [TODO 4]
# these two dictionaries below are very useful as the have prompt: pass_count pairs
# """
# pass_rate_dict, all_results = evaluator.evaluate_generations_generator(generator, prompts)


# # This computes the pass_at_k values. ks=[1,10,100] means we evaluate pass@1, pass@10, pass@100, n = NUM_GENERATIONS means how many generations we have per prompt
# pass_at_k_scores = evaluator.pass_at_k(pass_rate_dict, ks=[1], n=NUM_GENERATIONS)

# print("Pass counts:", pass_rate_dict)
# print("Num results:", len(all_results))

# for k in pass_at_k_scores:
#   print(f'pass@{k} is: {pass_at_k_scores[k]}')

In [ ]:
basic = """

// SPDX-License-Identifier: Apache-2.0
/* -*- P4_16 -*- */
#include <core.p4>
#include <v1model.p4>

const bit<16> TYPE_IPV4 = 0x800;

/*************************************************************************
*********************** H E A D E R S  ***********************************
* This program skeleton defines minimal Ethernet and IPv4 headers and    *
* a simple LPM (Longest-Prefix Match) IPv4 forwarding pipeline.          *
* The exercise intentionally leaves TODOs for learners to implement.     *
*************************************************************************/

typedef bit<9>  egressSpec_t;   // Standard BMv2 uses 9 bits for egress_spec
typedef bit<48> macAddr_t;      // Ethernet MAC address
typedef bit<32> ip4Addr_t;      // IPv4 address

header ethernet_t {
    macAddr_t dstAddr;
    macAddr_t srcAddr;
    bit<16>   etherType;
}

header ipv4_t {
    bit<4>    version;
    bit<4>    ihl;
    bit<8>    diffserv;
    bit<16>   totalLen;
    bit<16>   identification;
    bit<3>    flags;
    bit<13>   fragOffset;
    bit<8>    ttl;
    bit<8>    protocol;
    bit<16>   hdrChecksum;
    ip4Addr_t srcAddr;
    ip4Addr_t dstAddr;
}

struct metadata {
    /* empty */
}

struct headers {
    ethernet_t   ethernet;
    ipv4_t       ipv4;
}

/*************************************************************************
*********************** P A R S E R  *************************************
* New to P4? A typical parser does this:
*   start -> parse_ethernet
*   parse_ethernet:
*       if etherType == TYPE_IPV4 -> parse_ipv4
*       else accept
*   parse_ipv4 -> accept
* This skeleton leaves the actual states as a TODO to implement later.   *
*************************************************************************/

parser MyParser(packet_in packet,
                out headers hdr,
                inout metadata meta,
                inout standard_metadata_t standard_metadata) {

    state start {
        /* TODO: add parser logic
         * Suggested outline:
         *   1) Extract Ethernet: packet.extract(hdr.ethernet);
         *   2) If hdr.ethernet.etherType == TYPE_IPV4 -> parse IPv4
         *   3) Otherwise -> transition accept
         */
        transition accept;
    }
}


/*************************************************************************
************   C H E C K S U M    V E R I F I C A T I O N   *************
*************************************************************************/

control MyVerifyChecksum(inout headers hdr, inout metadata meta) {
    apply {  }
}


/*************************************************************************
**************  I N G R E S S   P R O C E S S I N G   *******************
* High-level intent:
*   - Do an LPM lookup on IPv4 dstAddr
*   - On hit, call ipv4_forward(next-hop MAC, output port)
*   - Otherwise, drop or NoAction (as configured)                         *
*************************************************************************/

control MyIngress(inout headers hdr,
                  inout metadata meta,
                  inout standard_metadata_t standard_metadata) {

    action drop() {
        mark_to_drop(standard_metadata);
    }

    /*********************************************************************
     * NOTE FOR NEW READERS:
     * 'ipv4_forward(dstAddr, port)' is invoked by table 'ipv4_lpm'.
     *
     * The values for 'dstAddr' and 'port' are *action data* supplied by
     * the control plane when it installs entries in 'ipv4_lpm'.
     *
     * They mean:
     *   - dstAddr  => Ethernet destination MAC for the next hop
     *   - port     => output port (ultimately written to standard_metadata.egress_spec)
     *
     * Example (BMv2 simple_switch_CLI):
     *   table_add ipv4_lpm ipv4_forward 10.0.1.1/32 => 00:00:00:00:01:00 1
     * which passes MAC=00:00:00:00:01:00 and PORT=1 as action parameters
     * into ipv4_forward(dstAddr, port).
     *********************************************************************/
    action ipv4_forward(macAddr_t dstAddr, egressSpec_t port) {
        /*
            Action function for forwarding IPv4 packets.

            TODO: Implement the forwarding steps, for example:
              - standard_metadata.egress_spec = port;
              - hdr.ethernet.dstAddr = dstAddr;
              - (optionally) set hdr.ethernet.srcAddr to the switch MAC for 'port'
              - adjust IPv4 TTL and checksums as needed
        */
    }

    /*********************************************************************
     * LPM table for IPv4:
     *   - Matches on hdr.ipv4.dstAddr using longest-prefix match (lpm)
     *   - On hit, calls ipv4_forward with *action data* populated by the
     *     control plane when it installs the table entry.
     *********************************************************************/
    table ipv4_lpm {
        key = {
            hdr.ipv4.dstAddr: lpm;
        }
        actions = {
            ipv4_forward;
            drop;
            NoAction;
        }
        size = 1024;
        default_action = NoAction();
    }

    apply {
        /* TODO: fix ingress control logic
         *  - Good practice: apply ipv4_lpm only when the IPv4 header is valid, e.g.:
         *      if (hdr.ipv4.isValid()) { ipv4_lpm.apply(); }
         *    This skeleton currently applies unconditionally for the exercise.
         */
        ipv4_lpm.apply();
    }
}

/*************************************************************************
****************  E G R E S S   P R O C E S S I N G   *******************
* Often used for queue marks, mirroring, or post-routing edits.          *
*************************************************************************/

control MyEgress(inout headers hdr,
                 inout metadata meta,
                 inout standard_metadata_t standard_metadata) {
    apply {  }
}

/*************************************************************************
*************   C H E C K S U M    C O M P U T A T I O N   **************
* This block shows how to compute IPv4 header checksum when needed.      *
*************************************************************************/

control MyComputeChecksum(inout headers hdr, inout metadata meta) {
     apply {
        update_checksum(
            hdr.ipv4.isValid(),
            { hdr.ipv4.version,
              hdr.ipv4.ihl,
              hdr.ipv4.diffserv,
              hdr.ipv4.totalLen,
              hdr.ipv4.identification,
              hdr.ipv4.flags,
              hdr.ipv4.fragOffset,
              hdr.ipv4.ttl,
              hdr.ipv4.protocol,
              hdr.ipv4.srcAddr,
              hdr.ipv4.dstAddr },
            hdr.ipv4.hdrChecksum,
            HashAlgorithm.csum16);
    }
}


/*************************************************************************
***********************  D E P A R S E R  *******************************
* The deparser serializes headers back onto the packet in order.         *
*************************************************************************/

control MyDeparser(packet_out packet, in headers hdr) {
    apply {
        /*
        Typical implementation (left as a TODO for learners):
            packet.emit(hdr.ethernet);
            packet.emit(hdr.ipv4);   // per P4_16 spec, emit appends a header
                                     // only if it is valid; no 'if' needed.
        */
    }
}

/*************************************************************************
***********************  S W I T C H  ***********************************
*************************************************************************/

V1Switch(
MyParser(),
MyVerifyChecksum(),
MyIngress(),
MyEgress(),
MyComputeChecksum(),
MyDeparser()
) main;

"""

In [ ]:
firewall = """

// SPDX-License-Identifier: Apache-2.0
/* -*- P4_16 -*- */
#include <core.p4>
#include <v1model.p4>

/* CONSTANTS */

const bit<16> TYPE_IPV4 = 0x800;
const bit<8>  TYPE_TCP  = 6;

#define BLOOM_FILTER_ENTRIES 4096
#define BLOOM_FILTER_BIT_WIDTH 1

/*************************************************************************
*********************** H E A D E R S  ***********************************
*************************************************************************/

typedef bit<9>  egressSpec_t;
typedef bit<48> macAddr_t;
typedef bit<32> ip4Addr_t;

header ethernet_t {
    macAddr_t dstAddr;
    macAddr_t srcAddr;
    bit<16>   etherType;
}

header ipv4_t {
    bit<4>    version;
    bit<4>    ihl;
    bit<8>    diffserv;
    bit<16>   totalLen;
    bit<16>   identification;
    bit<3>    flags;
    bit<13>   fragOffset;
    bit<8>    ttl;
    bit<8>    protocol;
    bit<16>   hdrChecksum;
    ip4Addr_t srcAddr;
    ip4Addr_t dstAddr;
}

header tcp_t{
    bit<16> srcPort;
    bit<16> dstPort;
    bit<32> seqNo;
    bit<32> ackNo;
    bit<4>  dataOffset;
    bit<4>  res;
    bit<1>  cwr;
    bit<1>  ece;
    bit<1>  urg;
    bit<1>  ack;
    bit<1>  psh;
    bit<1>  rst;
    bit<1>  syn;
    bit<1>  fin;
    bit<16> window;
    bit<16> checksum;
    bit<16> urgentPtr;
}

struct metadata {
    /* empty */
}

struct headers {
    ethernet_t   ethernet;
    ipv4_t       ipv4;
    tcp_t        tcp;
}

/*************************************************************************
*********************** P A R S E R  ***********************************
*************************************************************************/

parser MyParser(packet_in packet,
                out headers hdr,
                inout metadata meta,
                inout standard_metadata_t standard_metadata) {

    state start {
        transition parse_ethernet;
    }

    state parse_ethernet {
        packet.extract(hdr.ethernet);
        transition select(hdr.ethernet.etherType) {
            TYPE_IPV4: parse_ipv4;
            default: accept;
        }
    }

    state parse_ipv4 {
        packet.extract(hdr.ipv4);
        transition select(hdr.ipv4.protocol){
            TYPE_TCP: tcp;
            default: accept;
        }
    }

    state tcp {
       packet.extract(hdr.tcp);
       transition accept;
    }
}

/*************************************************************************
************   C H E C K S U M    V E R I F I C A T I O N   *************
*************************************************************************/

control MyVerifyChecksum(inout headers hdr, inout metadata meta) {
    apply {  }
}


/*************************************************************************
**************  I N G R E S S   P R O C E S S I N G   *******************
*************************************************************************/

control MyIngress(inout headers hdr,
                  inout metadata meta,
                  inout standard_metadata_t standard_metadata) {

    register<bit<BLOOM_FILTER_BIT_WIDTH>>(BLOOM_FILTER_ENTRIES) bloom_filter_1;
    register<bit<BLOOM_FILTER_BIT_WIDTH>>(BLOOM_FILTER_ENTRIES) bloom_filter_2;
    bit<32> reg_pos_one; bit<32> reg_pos_two;
    bit<1> reg_val_one; bit<1> reg_val_two;
    bit<1> direction;

    action drop() {
        mark_to_drop(standard_metadata);
    }

    action compute_hashes(ip4Addr_t ipAddr1, ip4Addr_t ipAddr2, bit<16> port1, bit<16> port2){
       //Get register position
       hash(reg_pos_one, HashAlgorithm.crc16, (bit<32>)0, {ipAddr1,
                                                           ipAddr2,
                                                           port1,
                                                           port2,
                                                           hdr.ipv4.protocol},
                                                           (bit<32>)BLOOM_FILTER_ENTRIES);

       hash(reg_pos_two, HashAlgorithm.crc32, (bit<32>)0, {ipAddr1,
                                                           ipAddr2,
                                                           port1,
                                                           port2,
                                                           hdr.ipv4.protocol},
                                                           (bit<32>)BLOOM_FILTER_ENTRIES);
    }

    action ipv4_forward(macAddr_t dstAddr, egressSpec_t port) {
        standard_metadata.egress_spec = port;
        hdr.ethernet.srcAddr = hdr.ethernet.dstAddr;
        hdr.ethernet.dstAddr = dstAddr;
        hdr.ipv4.ttl = hdr.ipv4.ttl - 1;
    }

    table ipv4_lpm {
        key = {
            hdr.ipv4.dstAddr: lpm;
        }
        actions = {
            ipv4_forward;
            drop;
            NoAction;
        }
        size = 1024;
        default_action = drop();
    }

    action set_direction(bit<1> dir) {
        direction = dir;
    }

    table check_ports {
        key = {
            standard_metadata.ingress_port: exact;
            standard_metadata.egress_spec: exact;
        }
        actions = {
            set_direction;
            NoAction;
        }
        size = 1024;
        default_action = NoAction();
    }

    apply {
        if (hdr.ipv4.isValid()){
            ipv4_lpm.apply();
            if (hdr.tcp.isValid()){
                direction = 0; // default
                if (check_ports.apply().hit) {
                    // test and set the bloom filter
                    if (direction == 0) {
                        compute_hashes(hdr.ipv4.srcAddr, hdr.ipv4.dstAddr, hdr.tcp.srcPort, hdr.tcp.dstPort);
                    }
                    else {
                        compute_hashes(hdr.ipv4.dstAddr, hdr.ipv4.srcAddr, hdr.tcp.dstPort, hdr.tcp.srcPort);
                    }
                    // Packet comes from internal network
                    if (direction == 0){
                        // If there is a syn we update the bloom filter and add the entry
                        if (hdr.tcp.syn == 1){
                            bloom_filter_1.write(reg_pos_one, 1);
                            bloom_filter_2.write(reg_pos_two, 1);
                        }
                    }
                    // Packet comes from outside
                    else if (direction == 1){
                        // Read bloom filter cells to check if there are 1's
                        bloom_filter_1.read(reg_val_one, reg_pos_one);
                        bloom_filter_2.read(reg_val_two, reg_pos_two);
                        // only allow flow to pass if both entries are set
                        if (reg_val_one != 1 || reg_val_two != 1){
                            drop();
                        }
                    }
                }
            }
        }
    }
}

/*************************************************************************
****************  E G R E S S   P R O C E S S I N G   *******************
*************************************************************************/

control MyEgress(inout headers hdr,
                 inout metadata meta,
                 inout standard_metadata_t standard_metadata) {
    apply {  }
}

/*************************************************************************
*************   C H E C K S U M    C O M P U T A T I O N   **************
*************************************************************************/

control MyComputeChecksum(inout headers  hdr, inout metadata meta) {
     apply {
        update_checksum(
            hdr.ipv4.isValid(),
            { hdr.ipv4.version,
              hdr.ipv4.ihl,
              hdr.ipv4.diffserv,
              hdr.ipv4.totalLen,
              hdr.ipv4.identification,
              hdr.ipv4.flags,
              hdr.ipv4.fragOffset,
              hdr.ipv4.ttl,
              hdr.ipv4.protocol,
              hdr.ipv4.srcAddr,
              hdr.ipv4.dstAddr },
            hdr.ipv4.hdrChecksum,
            HashAlgorithm.csum16);
    }
}

/*************************************************************************
***********************  D E P A R S E R  *******************************
*************************************************************************/

control MyDeparser(packet_out packet, in headers hdr) {
    apply {
        packet.emit(hdr.ethernet);
        packet.emit(hdr.ipv4);
        packet.emit(hdr.tcp);
    }
}

/*************************************************************************
***********************  S W I T C H  *******************************
*************************************************************************/

V1Switch(
MyParser(),
MyVerifyChecksum(),
MyIngress(),
MyEgress(),
MyComputeChecksum(),
MyDeparser()
) main;

"""

In [ ]:
""" [TODO 1] (done)
So the few_shot_examples array should be populated with FewShotExample objects that are of (intent, corresponding code) format.
you have calls to this class in other notebooks for pass@k, which are the exact examples you wanna use.

GPT -> REASONING
LLAMA, QWEN -> COMMENT REASONING OUT

"""
# prompt_builder = PromptBuilder(tokenizer=None, few_shot_examples=[FewShotExample("basic ip4 forwarding", code)])
prompt_builder = PromptBuilder(tokenizer=None, few_shot_examples = [
    FewShotExample("Drop all TCP SYN packets from a given blacklist of source IPs", """#include <core.p4>
#include <v1model.p4>

const bit<16> TYPE_IPV4 = 0x800;
const bit<8> TCP_PROTOCOL = 6;
const bit<6> TCP_SYN = 2;

typedef bit<9> egressSpec_t;
typedef bit<48> macAddr_t;
typedef bit<32> ip4Addr_t;

header ethernet_t {
    macAddr_t dstAddr;
    macAddr_t srcAddr;
    bit<16> etherType;
}

header ipv4_t {
    bit<4> version;
    bit<4> ihl;
    bit<8> diffserv;
    bit<16> totalLen;
    bit<16> identification;
    bit<3> flags;
    bit<13> fragOffset;
    bit<8> ttl;
    bit<8> protocol;
    bit<16> hdrChecksum;
    ip4Addr_t srcAddr;
    ip4Addr_t dstAddr;
}

header tcp_t {
    bit<16> srcPort;
    bit<16> dstPort;
    bit<32> seqNo;
    bit<32> ackNo;
    bit<4> dataOffset;
    bit<3> res;
    bit<3> ecn;
    bit<6> ctrl;
    bit<16> window;
    bit<16> checksum;
    bit<16> urgentPtr;
}

struct metadata {
    /* empty */
}

struct headers {
    ethernet_t ethernet;
    ipv4_t ipv4;
    tcp_t tcp;
}

parser MyParser(packet_in packet,
                out headers hdr,
                inout metadata meta,
                inout standard_metadata_t standard_metadata) {

    state start {
        transition parse_ethernet;
    }

    state parse_ethernet {
        packet.extract(hdr.ethernet);
        transition select(hdr.ethernet.etherType) {
            TYPE_IPV4: parse_ipv4;
            default: accept;
        }
    }

    state parse_ipv4 {
        packet.extract(hdr.ipv4);
        transition select(hdr.ipv4.protocol) {
            TCP_PROTOCOL: parse_tcp;
            default: accept;
        }
    }

    state parse_tcp {
        packet.extract(hdr.tcp);
        transition accept;
    }
}

control MyVerifyChecksum(inout headers hdr, inout metadata meta) {
    apply { }
}

control MyIngress(inout headers hdr,
                  inout metadata meta,
                  inout standard_metadata_t standard_metadata) {
    action drop() {
        mark_to_drop(standard_metadata);
    }

    table blacklist {
        key = {
            hdr.ipv4.srcAddr: exact;
        }
        actions = {
            drop;
            NoAction;
        }
        size = 1024;
        default_action = NoAction();
    }

    apply {
        if (hdr.ipv4.isValid() && hdr.tcp.isValid()) {
            if (hdr.tcp.ctrl & TCP_SYN != 0) {
                blacklist.apply();
            }
        }
    }
}

control MyEgress(inout headers hdr,
                 inout metadata meta,
                 inout standard_metadata_t standard_metadata) {
    apply { }
}

control MyComputeChecksum(inout headers hdr, inout metadata meta) {
    apply {
        update_checksum(
            hdr.ipv4.isValid(),
            { hdr.ipv4.version,
              hdr.ipv4.ihl,
              hdr.ipv4.diffserv,
              hdr.ipv4.totalLen,
              hdr.ipv4.identification,
              hdr.ipv4.flags,
              hdr.ipv4.fragOffset,
              hdr.ipv4.ttl,
              hdr.ipv4.protocol,
              hdr.ipv4.srcAddr,
              hdr.ipv4.dstAddr },
            hdr.ipv4.hdrChecksum,
            HashAlgorithm.csum16);
    }
}

control MyDeparser(packet_out packet, in headers hdr) {
    apply {
        packet.emit(hdr.ethernet);
        packet.emit(hdr.ipv4);
        packet.emit(hdr.tcp);
    }
}

V1Switch(
    MyParser(),
    MyVerifyChecksum(),
    MyIngress(),
    MyEgress(),
    MyComputeChecksum(),
    MyDeparser()
) main;"""),
    FewShotExample("Distribute packets evenly across 4 ports using an ECMP hash of the five‑tuple", """#include <core.p4>
#include <v1model.p4>

typedef bit<9> egressSpec_t;
typedef bit<48> macAddr_t;
typedef bit<32> ip4Addr_t;

header ethernet_t {
    macAddr_t dstAddr;
    macAddr_t srcAddr;
    bit<16> etherType;
}

header ipv4_t {
    bit<4> version;
    bit<4> ihl;
    bit<8> diffserv;
    bit<16> totalLen;
    bit<16> identification;
    bit<3> flags;
    bit<13> fragOffset;
    bit<8> ttl;
    bit<8> protocol;
    bit<16> hdrChecksum;
    ip4Addr_t srcAddr;
    ip4Addr_t dstAddr;
}

header tcp_t {
    bit<16> srcPort;
    bit<16> dstPort;
    bit<32> seqNo;
    bit<32> ackNo;
    bit<4> dataOffset;
    bit<3> res;
    bit<3> ecn;
    bit<6> ctrl;
    bit<16> window;
    bit<16> checksum;
    bit<16> urgentPtr;
}

struct metadata {
    bit<2> ecmp_select;
}

struct headers {
    ethernet_t ethernet;
    ipv4_t ipv4;
    tcp_t tcp;
}

parser MyParser(packet_in packet,
                out headers hdr,
                inout metadata meta,
                inout standard_metadata_t standard_metadata) {

    state start {
        transition parse_ethernet;
    }

    state parse_ethernet {
        packet.extract(hdr.ethernet);
        transition select(hdr.ethernet.etherType) {
            0x800: parse_ipv4;
            default: accept;
        }
    }

    state parse_ipv4 {
        packet.extract(hdr.ipv4);
        transition select(hdr.ipv4.protocol) {
            6: parse_tcp;
            default: accept;
        }
    }

    state parse_tcp {
        packet.extract(hdr.tcp);
        transition accept;
    }
}

control MyVerifyChecksum(inout headers hdr, inout metadata meta) {
    apply { }
}

control MyIngress(inout headers hdr,
                    inout metadata meta,
                    inout standard_metadata_t standard_metadata) {
    action drop() {
        mark_to_drop(standard_metadata);
    }

    action set_ecmp_select() {
        hash(meta.ecmp_select,
             HashAlgorithm.crc16,
             (bit<16>)0,
             { hdr.ipv4.srcAddr,
               hdr.ipv4.dstAddr,
               hdr.ipv4.protocol,
               hdr.tcp.srcPort,
               hdr.tcp.dstPort },
             (bit<16>)4);
    }

    action set_nhop(macAddr_t dstAddr, egressSpec_t port) {
        hdr.ethernet.dstAddr = dstAddr;
        standard_metadata.egress_spec = port;
        hdr.ipv4.ttl = hdr.ipv4.ttl - 1;
    }

    table ecmp_group {
        key = {
            hdr.ipv4.dstAddr: lpm;
        }
        actions = {
            set_ecmp_select;
            drop;
        }
        size = 1024;
    }

    table ecmp_nhop {
        key = {
            meta.ecmp_select: exact;
        }
        actions = {
            set_nhop;
            drop;
        }
        size = 4;
    }

    apply {
        if (hdr.ipv4.isValid() && hdr.tcp.isValid()) {
            ecmp_group.apply();
            ecmp_nhop.apply();
        }
    }
}

control MyEgress(inout headers hdr,
                 inout metadata meta,
                 inout standard_metadata_t standard_metadata) {
    apply { }
}

control MyComputeChecksum(inout headers hdr, inout metadata meta) {
    apply {
        update_checksum(
            hdr.ipv4.isValid(),
            { hdr.ipv4.version,
              hdr.ipv4.ihl,
              hdr.ipv4.diffserv,
              hdr.ipv4.totalLen,
              hdr.ipv4.identification,
              hdr.ipv4.flags,
              hdr.ipv4.fragOffset,
              hdr.ipv4.ttl,
              hdr.ipv4.protocol,
              hdr.ipv4.srcAddr,
              hdr.ipv4.dstAddr },
            hdr.ipv4.hdrChecksum,
            HashAlgorithm.csum16);
    }
}

control MyDeparser(packet_out packet, in headers hdr) {
    apply {
        packet.emit(hdr.ethernet);
        packet.emit(hdr.ipv4);
        packet.emit(hdr.tcp);
    }
}

V1Switch(
    MyParser(),
    MyVerifyChecksum(),
    MyIngress(),
    MyEgress(),
    MyComputeChecksum(),
    MyDeparser()
) main;""")
]
)


# "google/gemini-2.5-flash",
available_models = [" PLACEHOLDER, CHOOSE AN ACTUAL ENTRY ", "openai/gpt-5-mini", "meta-llama/llama-4-maverick", "qwen/qwen3-max"]

MODEL_NAME = available_models[1]

gen = OpenRouterGenerator(prompt_builder, model_name=MODEL_NAME, token="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")
evaluator = Evaluator()

prompts = list(validation_ds["annotation"])

NUM_GENERATIONS = 10 # how many generations per prompt

generator = gen._async_generate(prompts , n=NUM_GENERATIONS, batch_size=32)

""" [TODO 4]
these two dictionaries above are very useful as the have prompt: pass_count pairs
"""
pass_rate_dict, all_results = evaluator.evaluate_generations_generator(generator, prompts)

# This computes the pass_at_k values. ks=[1,10,100] means we evaluate pass@1, pass@10, pass@100, n = NUM_GENERATIONS means how many generations we have per prompt
pass_at_k_scores = evaluator.pass_at_k(pass_rate_dict, ks=[1,10], n=NUM_GENERATIONS)



print("Pass counts:", pass_rate_dict)
print("Num results:", len(all_results))

for k in pass_at_k_scores:
  print(f'pass@{k} is: {pass_at_k_scores[k]}')

compile_rate = evaluator.compile_rate_at_1(all_results)
print(f"Compile rate: {compile_rate}")

Validating: 0gen [00:00, ?gen/s]

generating...:   0%|          | 0/410 [00:00<?, ?it/s]

Generation failed, but status == 200 with 200. Retrying 1/20
Pass counts: {'Swap Ethernet MAC addresses and mirror packets to the same port.': 0, 'Performs IPv4 routing with tunneling and TCP parsing': 0, 'Performs IPv4 routing using exact and LPM tables, decrementing TTL and updating checksum.': 1, 'Forwards packets with custom headers based on exact matches of source and destination MAC addresses and custom header fields.': 1, 'Performs Ethernet frame processing, including MAC address lookup, port setting, and header field modifications.': 1, 'Monitors DNS traffic and tracks packet counts and byte counts for known domains.': 0, 'Tunnel ingress/egress processing and IPv4 forwarding': 0, 'IPv4 packet forwarding with FIB lookup and MAC rewriting': 2, "Validate and process a custom header with a 32-bit field 'a'.": 0, 'Forwards IPv4 packets based on destination IP address and updates Ethernet and IP headers.': 0, 'Route packets based on Ethernet type and custom header fields, modifying s

In [ ]:
import pickle
model_name_modified = MODEL_NAME.replace("/", "")

with open(f"{model_name_modified}_data.pkl", "wb") as f:
    pickle.dump({"pass_rate_dict": pass_rate_dict, "all_results": all_results}, f)

In [ ]:
import pickle
# model_name_modified = "meta-llamallama-4-maverick"
with open(f'{model_name_modified}_data.pkl', 'rb') as f:
        # Load the data
        d = pickle.load(f)

In [ ]:
evaluator = Evaluator()
